# Enterprise AI Knowledge & Data Assistant

## Problem Statement

TechNova Technologies is a software company whose information is distributed
across different sources.

The company has:

- Internal company documents containing policies and organizational information
- External information available through Wikipedia and Web Search
- Employee and sales information stored in a SQL database

Employees currently need to search these sources separately.

The company wants to build an AI-powered Enterprise Assistant using
LangChain and OpenRouter.

## Objective

The AI agent should be able to:

1. Retrieve information from internal company documents.
2. Search external knowledge when required.
3. Query structured information from a SQL database.
4. Automatically select the appropriate tool.
5. Combine information from multiple tools when required.
6. Generate a natural-language answer.

---
## System Architecture

<p align="center">

<b>User</b>
<br>
↓
<br>
<b>LangChain AI Agent</b>
<br>
↓
<br>
<b>OpenRouter LLM</b>
<br>
↓
<br>
<b>Intelligent Tool Selection</b>

</p>

<table align="center">
<tr>
<td align="center" width="33%">

<b>Retrieval Tool</b>
<br><br>
↓
<br><br>
<b>Chroma Vector DB</b>
<br><br>
↓
<br><br>
<b>Company Documents</b>

</td>

<td align="center" width="33%">

<b>External Search Tool</b>
<br><br>
↓
<br><br>
<b>Wikipedia / Web Search</b>
<br><br>
↓
<br><br>
<b>External Knowledge</b>

</td>

<td align="center" width="33%">

<b>Database Tool</b>
<br><br>
↓
<br><br>
<b>SQLite Database</b>
<br><br>
↓
<br><br>
<b>Company Data</b>

</td>
</tr>
</table>

<p align="center">

↓
<br>
<b>Tool Results</b>
<br>
↓
<br>
<b>OpenRouter LLM</b>
<br>
↓
<br>
<b>Final Natural-Language Answer</b>

</p>

---

## Key Concept

<p align="center">

<b>Agentic AI = LLM + Tools + Decision Making + External Data</b>

</p>

In [27]:
# Install required libraries

! pip install -U langchain langchain-openrouter langchain-community langchain-chroma chromadb sentence-transformers wikipedia


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in .env file")

print("OpenRouter API key loaded successfully.")

OpenRouter API key loaded successfully.


In [29]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="z-ai/glm-5.3-flash",
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    temperature=0
)

print("Model initialized successfully")

Model initialized successfully


In [30]:
response = model.invoke("Explain what an AI agent is in one sentence.")

print(response.content)

An AI agent is a software system that can autonomously perceive its environment, make decisions, and take actions to achieve specific goals, typically with minimal human intervention.


In [48]:
from langchain_core.documents import Document

# TechNova internal company documents

documents = [
    Document(
        page_content="""
        TechNova Technologies is a software company specializing in
        Artificial Intelligence, Cloud Computing, Cybersecurity,
        and Software Development.

        The company has offices in Hyderabad, Bangalore, and Delhi.
        """
    ),

    Document(
        page_content="""
        TechNova Employee Work From Home Policy:

        Employees can work remotely up to 2 days per week.
        Remote work requires approval from the employee's manager.
        Employees must remain available during official working hours.
        """
    ),

    Document(
        page_content="""
        TechNova Employee Benefits:

        Employees receive 24 annual paid leaves.
        The company provides an annual learning budget of
        30000 INR for approved technical courses, conferences,
        and professional development programs.
        """
    ),

    Document(
        page_content="""
        TechNova Working Hours:

        Standard working hours are from 9:30 AM to 6:30 PM.
        Employees are expected to complete their assigned working hours.
        Flexible working arrangements require manager approval.
        """
    )
]

print(f"Number of documents: {len(documents)}")

Number of documents: 4


In [32]:
!pip install langchain-chroma chromadb


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Create the embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create Chroma vector database
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="technova_knowledge"
)

print("Chroma vector database created successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3462.62it/s]


Chroma vector database created successfully!


In [34]:
from langchain_chroma import Chroma

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

In [35]:
results = retriever.invoke(
    "What is TechNova's work from home policy?"
)

for doc in results:
    print(doc.page_content)


        TechNova Employee Work From Home Policy:

        Employees can work remotely up to 2 days per week.
        Remote work requires approval from the employee's manager.
        Employees must remain available during official working hours.
        

        TechNova Employee Work From Home Policy:

        Employees can work remotely up to 2 days per week.
        Remote work requires approval from the employee's manager.
        Employees must remain available during official working hours.
        


In [36]:
from langchain_core.tools import tool

@tool
def search_company_documents(query: str) -> str:
    """
    Search TechNova's internal company documents.

    Use this tool for company policies, working hours,
    employee benefits, offices, and other internal information.
    """
    
    docs = retriever.invoke(query)
    
    return "\n\n".join(
        doc.page_content for doc in docs
    )

print("Retrieval tool created successfully!")

Retrieval tool created successfully!


In [37]:
result = search_company_documents.invoke({
    "query": "What is TechNova's work from home policy?"
})

print(result)


        TechNova Employee Work From Home Policy:

        Employees can work remotely up to 2 days per week.
        Remote work requires approval from the employee's manager.
        Employees must remain available during official working hours.
        


        TechNova Employee Work From Home Policy:

        Employees can work remotely up to 2 days per week.
        Remote work requires approval from the employee's manager.
        Employees must remain available during official working hours.
        


In [38]:
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

# Create Wikipedia API wrapper
wiki_wrapper = WikipediaAPIWrapper(
    top_k_results=2,
    doc_content_chars_max=3000
)

# Create LangChain Wikipedia tool
wikipedia_tool = WikipediaQueryRun(
    api_wrapper=wiki_wrapper
)

print("Wikipedia tool created successfully!")

Wikipedia tool created successfully!


In [39]:
result = wikipedia_tool.invoke("Artificial Intelligence")

print(result)

Page: Artificial intelligence
Summary: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g. images, audio, and videos).
The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics. To reach these goals, AI researchers use techniques 

In [40]:
import sqlite3



conn = sqlite3.connect("technova.db")
cursor = conn.cursor()



cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    salary INTEGER
)
""")



cursor.execute("""
CREATE TABLE IF NOT EXISTS sales (
    id INTEGER PRIMARY KEY,
    product TEXT,
    region TEXT,
    amount INTEGER
)
""")



employees = [
    (1, "Rahul Sharma", "AI", 85000),
    (2, "Priya Singh", "Data Science", 92000),
    (3, "Arjun Kumar", "Software", 78000),
    (4, "Sneha Patel", "HR", 65000),
    (5, "Vikram Reddy", "AI", 95000),
    (6, "Ananya Das", "Finance", 72000),
    (7, "Rohit Verma", "Software", 88000),
    (8, "Meera Nair", "Data Science", 90000),
    (9, "Karan Gupta", "Sales", 70000),
    (10, "Neha Joshi", "AI", 87000)
]

cursor.executemany("""
INSERT OR REPLACE INTO employees
(id, name, department, salary)
VALUES (?, ?, ?, ?)
""", employees)



sales = [
    (1, "Laptop", "North", 120000),
    (2, "Smartphone", "South", 95000),
    (3, "Tablet", "East", 75000),
    (4, "Laptop", "West", 140000),
    (5, "Smartwatch", "North", 55000),
    (6, "Smartphone", "East", 110000),
    (7, "Laptop", "South", 135000),
    (8, "Tablet", "West", 82000),
    (9, "Smartwatch", "East", 62000),
    (10, "Laptop", "North", 155000),
    (11, "Smartphone", "West", 105000),
    (12, "Tablet", "South", 78000),
    (13, "Smartwatch", "North", 59000),
    (14, "Laptop", "East", 130000),
    (15, "Smartphone", "North", 125000)
]

cursor.executemany("""
INSERT OR REPLACE INTO sales
(id, product, region, amount)
VALUES (?, ?, ?, ?)
""", sales)



conn.commit()

print("SQLite database created and sample data inserted successfully!")

conn.close()

SQLite database created and sample data inserted successfully!


In [41]:
import sqlite3

conn = sqlite3.connect("technova.db")

cursor = conn.cursor()

print("EMPLOYEES")
print("=" * 50)

cursor.execute("SELECT * FROM employees")

for row in cursor.fetchall():
    print(row)

print("\nSALES")
print("=" * 50)

cursor.execute("SELECT * FROM sales")

for row in cursor.fetchall():
    print(row)

conn.close()

EMPLOYEES
(1, 'Rahul Sharma', 'AI', 85000)
(2, 'Priya Singh', 'Data Science', 92000)
(3, 'Arjun Kumar', 'Software', 78000)
(4, 'Sneha Patel', 'HR', 65000)
(5, 'Vikram Reddy', 'AI', 95000)
(6, 'Ananya Das', 'Finance', 72000)
(7, 'Rohit Verma', 'Software', 88000)
(8, 'Meera Nair', 'Data Science', 90000)
(9, 'Karan Gupta', 'Sales', 70000)
(10, 'Neha Joshi', 'AI', 87000)

SALES
(1, 'Laptop', 'North', 120000)
(2, 'Smartphone', 'South', 95000)
(3, 'Tablet', 'East', 75000)
(4, 'Laptop', 'West', 140000)
(5, 'Smartwatch', 'North', 55000)
(6, 'Smartphone', 'East', 110000)
(7, 'Laptop', 'South', 135000)
(8, 'Tablet', 'West', 82000)
(9, 'Smartwatch', 'East', 62000)
(10, 'Laptop', 'North', 155000)
(11, 'Smartphone', 'West', 105000)
(12, 'Tablet', 'South', 78000)
(13, 'Smartwatch', 'North', 59000)
(14, 'Laptop', 'East', 130000)
(15, 'Smartphone', 'North', 125000)


In [42]:
! pip install -q langchain langchain-community sqlalchemy


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [43]:
from langchain_community.utilities import SQLDatabase

# Connect LangChain to SQLite database
db = SQLDatabase.from_uri("sqlite:///technova.db")

print("Database connected successfully!")

Database connected successfully!


In [44]:
print(db.get_usable_table_names())

['employees', 'sales']


In [45]:
print(db.get_table_info())


CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	department TEXT, 
	salary INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	department	salary
1	Rahul Sharma	AI	85000
2	Priya Singh	Data Science	92000
3	Arjun Kumar	Software	78000
*/


CREATE TABLE sales (
	id INTEGER, 
	product TEXT, 
	region TEXT, 
	amount INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from sales table:
id	product	region	amount
1	Laptop	North	120000
2	Smartphone	South	95000
3	Tablet	East	75000
*/


In [46]:
from langchain_community.agent_toolkits import create_sql_agent

agent = create_sql_agent(
    llm=model,
    db=db,
    agent_type="tool-calling",
    verbose=True
)

print("SQL Agent created successfully!")

SQL Agent created successfully!


In [47]:
question = "Who is the highest paid employee?"

response = agent.invoke({
    "input": question
})

print(response["output"])



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


employees, sales
Invoking: `sql_db_schema` with `{'table_names': 'employees'}`



CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	department TEXT, 
	salary INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	department	salary
1	Rahul Sharma	AI	85000
2	Priya Singh	Data Science	92000
3	Arjun Kumar	Software	78000
*/
Invoking: `sql_db_query_checker` with `{'query': 'SELECT name, department, salary FROM employees ORDER BY salary DESC LIMIT 1'}`


```sql
SELECT name, department, salary FROM employees ORDER BY salary DESC LIMIT 1
```
Invoking: `sql_db_query` with `{'query': 'SELECT name, department, salary FROM employees ORDER BY salary DESC LIMIT 1'}`


[('Vikram Reddy', 'AI', 95000)]The highest paid employee is **Vikram Reddy**, who works in the **AI** department with a salary of **$95,000**.

> Finished chain.
The highest paid employee is **Vikram Reddy**, who wo